<a href="https://colab.research.google.com/github/calmrocks/ai-engineer-notebooks/blob/main/08-operations/01-observability-and-llmops.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Observability & LLMOps

**Goal:** Instrument an LLM app so you can see what it's actually doing in production: trace every call, log it safely, track cost/latency/errors, and catch quality drift before a user reports it.

Part of [ai-engineer-notebooks](https://github.com/calmrocks/ai-engineer-notebooks), the hands-on companion to the [FDE / AI Engineer transition plan](https://www.calm.rocks/resources/career-development/transition-fde-ai-engineer/).

## Setup

Each notebook is self-contained, so the next two cells stand it up from scratch:

1. **Install dependencies.** The `aien` package (this repo) carries the shared setup helper and pulls in the `groq` client, the only dependency this notebook needs.
2. **Load your API key.** Free key at [console.groq.com](https://console.groq.com/) (no credit card); in Colab add it via the **key icon** → **Add new secret** named exactly `GROQ_API_KEY`, notebook access on. Locally, set `GROQ_API_KEY` in your environment.

(Full walkthrough: [00-setup/00-environment.ipynb](https://colab.research.google.com/github/calmrocks/ai-engineer-notebooks/blob/main/00-setup/00-environment.ipynb).)

In [ ]:
%pip install -q "git+https://github.com/calmrocks/ai-engineer-notebooks.git"

In [ ]:
from aien import setup

# Loads GROQ_API_KEY and returns a ready Groq client.
client, MODEL = setup()

## Why observability is the job, not an add-on

Evals (section 04) tell you the system is good *before* you ship. Observability tells you it's *still* good after, on real traffic you never anticipated. LLM apps fail in ways ordinary services don't: no exception is raised when the model quietly starts giving worse answers, drifts off-topic, or a prompt change tanks quality for one class of users. The only way you find out is if you **instrumented it to tell you.**

The core unit is the **trace**: a structured record of one interaction — inputs, outputs, token usage, latency, and enough metadata to slice by later. Everything in LLMOps is built on having good traces: cost dashboards, latency SLOs, drift detection, and the feedback loop back into your eval set. We'll build a minimal tracer from scratch. The concept is what transfers; hosted tools just make it pretty.

## A minimal tracer

No framework needed. A trace is a dict per call with the fields you'll want to query later. The discipline is: capture it on *every* call, success or failure, and never let tracing throw.

In [ ]:
import time, json, uuid

TRACES = []   # in a real app this is a DB / logging pipeline, not a list

def traced_call(messages, tag=None, **kwargs):
    """Wrap a Groq call and record a structured trace. Never raises from tracing."""
    trace = {"id": str(uuid.uuid4())[:8], "tag": tag, "model": kwargs.get("model", MODEL)}
    t0 = time.monotonic()
    try:
        resp = client.chat.completions.create(model=kwargs.pop("model", MODEL),
                                               messages=messages, **kwargs)
        trace["latency_s"] = round(time.monotonic() - t0, 3)
        trace["ok"] = True
        trace["prompt_tokens"] = resp.usage.prompt_tokens
        trace["completion_tokens"] = resp.usage.completion_tokens
        trace["output"] = resp.choices[0].message.content
        return resp
    except Exception as e:
        trace["latency_s"] = round(time.monotonic() - t0, 3)
        trace["ok"] = False
        trace["error"] = f"{type(e).__name__}: {e}"
        raise
    finally:
        TRACES.append(trace)   # captured whether the call succeeded or blew up

# Drive some traffic
for q in ["What is TCP?", "Explain DNS in one sentence.", "What port does HTTPS use?"]:
    traced_call([{"role": "user", "content": q}], tag="demo", max_tokens=80)

print(json.dumps(TRACES[-1], indent=2)[:400])

## Log prompts safely: logging is a disclosure path (ties to security)

The obvious move, logging the full prompt and output, is also a **PII/secrets leak waiting to happen** (OWASP LLM02, section 07). Prompts often contain user data; outputs sometimes echo it. Careless logging quietly ships that to wherever your logs go. Two rules:

- **Redact before you log.** Mask emails, phone numbers, card-like digit runs.
- **Truncate and sample.** You rarely need the full text of every call; store a preview + hash, keep full payloads only for a sampled fraction or for errors.

In [ ]:
import re, hashlib

def redact(text):
    text = re.sub(r'[\w.+-]+@[\w-]+\.[\w.-]+', '<email>', text)
    text = re.sub(r'\b(?:\d[ -]?){13,16}\b', '<card>', text)
    text = re.sub(r'\b\d{3}[-.]?\d{3}[-.]?\d{4}\b', '<phone>', text)
    return text

def safe_log(text, keep=120):
    red = redact(text)
    return {"preview": red[:keep], "chars": len(text),
            "sha1": hashlib.sha1(text.encode()).hexdigest()[:10]}  # hash to dedup/trace w/o storing raw

sample = "Contact me at jane.doe@acme.com or 555-123-4567 about card 4111 1111 1111 1111."
print(safe_log(sample))

The log line now carries a redacted preview, a length, and a hash you can use to group identical prompts, enough to debug and analyze without turning your log store into a PII spill. (Full payloads, when you truly need them, go behind access controls and a retention window.)

## The metrics that matter

From a pile of traces, four numbers tell you whether the system is healthy. Compute them continuously and alert on deltas, not absolutes:

- **Cost.** Tokens × rate, aggregated (the math from `01-model-apis/04`). Watch for creep: a prompt tweak that adds 2k tokens × millions of calls is a budget event.
- **Latency.** Track p50 *and* p95/p99. Averages hide the tail that actually annoys users; TTFT matters for streamed UIs (`01-model-apis/03`).
- **Error/failure rate.** API errors, timeouts, *and* schema-validation failures (a call that "succeeded" but returned unparseable JSON is a failure to your app).
- **Quality proxies.** You can't run the full eval on live traffic, but cheap signals track drift: refusal rate, output length distribution, % of tool calls that validated, thumbs-up/down if you collect it.

In [ ]:
import statistics

def report(traces):
    ok = [t for t in traces if t.get("ok")]
    n = len(traces)
    lat = sorted(t["latency_s"] for t in ok)
    def pct(p):
        return lat[min(len(lat)-1, int(len(lat)*p))] if lat else 0
    tokens = sum(t.get("prompt_tokens",0)+t.get("completion_tokens",0) for t in ok)
    print(f"calls            : {n}")
    print(f"error rate       : {(n-len(ok))/n:.0%}" if n else "n/a")
    print(f"latency p50 / p95: {pct(0.5):.3f}s / {pct(0.95):.3f}s")
    print(f"total tokens     : {tokens:,}")
    # illustrative cost: $0.90 / 1M tokens (see 01-model-apis/04 for the real math)
    print(f"est. cost        : ${tokens * 0.90 / 1_000_000:.5f}")

report(TRACES)

## Closing the loop: observability → evals

The payoff of tracing isn't the dashboard. It's the **feedback loop**. Production surfaces inputs you never imagined, and each bad one is a new eval case:

1. A trace looks wrong (low quality proxy, user thumbs-down, a support complaint).
2. You pull that exact input from the traces.
3. It becomes a case in your golden set (`04-evals/01`) with the answer it *should* have given.
4. Your regression harness (`04-evals/03`) now guards against it forever.

That's the whole LLMOps flywheel: **eval before ship → observe in prod → feed failures back into evals.** Every bug becomes a test, the same discipline as backend regression testing, just applied to model behavior.

**Where the tools come in:** hosted platforms (**LangSmith, Langfuse, Arize Phoenix, Braintrust, Helicone**) give you this tracer, redaction, dashboards, and trace-to-eval wiring out of the box, plus trend history across weeks. Adopt one when the `TRACES = []` list stops scaling (more than one person needs to see the data). You'll evaluate them well precisely because you built the minimal version here.

## Practices & anti-patterns

| ✅ Do | ❌ Anti-pattern |
|---|---|
| Trace every call (in/out/tokens/latency/status), success *or* failure | Instrument nothing and find out from a customer complaint |
| Redact before logging; sample full payloads | Log raw prompts/outputs: a quiet PII/secrets spill |
| Track p50 **and** p95/p99; alert on deltas | Watch the mean only; miss the tail users actually feel |
| Track quality proxies (refusal rate, length, validation %) for drift | Assume "no exceptions" means quality is fine |
| Feed production failures back into the golden set (the flywheel) | Treat dashboards as the end goal; never close the loop |
| Adopt a hosted platform when the trace list stops scaling | Reach for LangSmith/Langfuse before you know what it automates |

(The full stack-wide list lives in [docs/best-practices-and-anti-patterns.md](https://github.com/calmrocks/ai-engineer-notebooks/blob/main/docs/best-practices-and-anti-patterns.md).)

## Exercises

1. **Tag and slice.** Add a `user_tier` field to each trace ("free"/"paid") and compute latency p95 and error rate per tier. Uneven pain across segments is invisible in a global average, which is why you slice.
2. **Alert on a delta.** Write `check_drift(window_a, window_b)` that flags when mean output length or refusal rate shifts more than X% between two batches of traces. This is drift detection in ~10 lines.
3. **Trace a RAG call end to end.** Instrument the `answer()` function from `03-rag` so one trace captures the query, retrieved doc ids, *and* the generation. Then when an answer is bad you can tell retrieval-fault from generation-fault (the `03-rag/04-why-rag-fails` distinction) straight from the log.
4. **Sampling policy.** Modify `traced_call` to store the full payload for 100% of errors but only 10% of successes. Cost of storing everything vs debugging power of storing nothing: where's your line, and why?